# To perform and visualize analyses online 

## Before the experiment: get everything ready 

### imports

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import hydra
from hydra.utils import get_original_cwd
import os
from omegaconf import DictConfig, OmegaConf
from dataclasses import dataclass
from typing import List, Dict, Any

from IPython.display import display



In [3]:
# Load config
import sys
import os
from pathlib import Path

# 

# Add the parent directory to the path so we can import modules properly
cwd = Path.cwd()
print(f"home directory: {cwd}")
relative_repo_path = "GitRepos/simulation_closed_loop"

# append repo path 
sys.path.append(str(cwd / relative_repo_path))

# Import Hydra config utilities
from omegaconf import DictConfig, OmegaConf
import hydra
from hydra.utils import instantiate
from hydra.core.config_store import ConfigStore
from hydra import compose, initialize

# Initialize Hydra with the relative path to the config directory
config_path = os.path.join(relative_repo_path,"config")
print(f"Config path: {config_path}")

# Initialize Hydra
with initialize(version_base="1.3", config_path=config_path):
    # Compose the configuration
    cfg = compose(config_name="config")

# Print the config to verify it loaded correctly
print("Configuration loaded successfully:")
print(OmegaConf.to_yaml(cfg))



home directory: /gpfs01/euler/User/ssuhai
Config path: GitRepos/simulation_closed_loop/config
Configuration loaded successfully:
data_subfolders:
  day: 20250717
  experiment: 1
DJ:
  username: ssuhai
  userinfo:
    experimenter: closedlooptest
    animal_loc: 1
    region_loc: 2
    field_loc: 3
    stimulus_loc: 4
    cond1_loc: 5
    data_dir: /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/data/recordings/first_closed_loop_experiment
  table_parameters:
    PreprocessParams:
      fs_resample: 60
    Stimulus:
      noise:
        stim_name: densenoise
        stim_family: noise
        pix_n_x: 20
        pix_n_y: 15
        skip_duplicates: true
        pix_scale_x_um: 40
        pix_scale_y_um: 40
        framerate: 5
    DNoiseTraceParams:
      dnoise_params_id: 1
      fupsample_trace: 20
      fupsample_stim: 4
      ref_time: stim
      fit_kind: gradient
      skip_duplicates: true
      pre_blur_sigma_s: 0.0
      post_blur_sigma_s: 0.0
paths:
  repo_directory:

In [4]:


from simulations.loop_components.dj_wrappers import DJTableHolder,Preprocessor,QualityAndTypeWrapper,STAWrapper
from simulations.loop_components.stimulus_file_copier import copy_stim_files,create_directory_structure



### Create processing components (connect them to DB)

In [5]:
# create preprocessor
os.environ["DJ_SUPPORT_FILEPATH_MANAGEMENT"] = "TRUE"

dj_table_holder = DJTableHolder(
                username=cfg.DJ.username, # type: ignore
                
                #paths
                home_directory=cfg.paths.home_directory, # type: ignore
                repo_directory=cfg.paths.repo_directory, # type: ignore
                dj_config_directory= cfg.paths.dj_config_directory, # type: ignore
                rgc_output_directory= cfg.paths.rgc_output_directory, # type: ignore
                data_subfolders=cfg.data_subfolders, # type: ignore


                userinfo= cfg.DJ.userinfo, # type: ignore

                table_parameters=cfg.DJ.table_parameters, # type: ignore

                # from overall configs
                debug=cfg.debug, # type: ignore
                plot_results=cfg.plot_results, # type: ignore

                    )

preprocessor = Preprocessor(dj_table_holder=dj_table_holder)




quality_type_analysis_wrapper = QualityAndTypeWrapper(
    dj_table_holder=dj_table_holder,)

sta_wrapper = STAWrapper(
    dj_table_holder=dj_table_holder,)

In [ ]:

# # Load config and tables
# dj_table_holder.load_config()
# dj_table_holder.load_tables()
# print(" loaded and configured successfully")


#dj_table_holder.clear_tables("all")
dj_table_holder.setup()




[2025-07-30 12:18:20,792][INFO]: Deleting 3 rows from `ageuler_ssuhai_closed_loop`.`__field__stack_averages`
[2025-07-30 12:18:20,862][INFO]: Deleting 3 rows from `ageuler_ssuhai_closed_loop`.`__presentation__scan_info`
[2025-07-30 12:18:20,909][INFO]: Deleting 9 rows from `ageuler_ssuhai_closed_loop`.`__presentation__stack_averages`
[2025-07-30 12:18:21,047][INFO]: Deleting 87 rows from `ageuler_ssuhai_closed_loop`.`__peak_s_t_a_position`
[2025-07-30 12:18:21,097][INFO]: Deleting 87 rows from `ageuler_ssuhai_closed_loop`.`__s_t_a__data_set`
[2025-07-30 12:18:21,149][INFO]: Deleting 87 rows from `ageuler_ssuhai_closed_loop`.`__split_r_f`
[2025-07-30 12:18:21,186][INFO]: Deleting 87 rows from `ageuler_ssuhai_closed_loop`.`__s_t_a`
[2025-07-30 12:18:21,219][INFO]: Deleting 87 rows from `ageuler_ssuhai_closed_loop`.`__d_noise_trace`
[2025-07-30 12:18:21,357][INFO]: Deleting 88 rows from `ageuler_ssuhai_closed_loop`.`__celltype_assignment`
[2025-07-30 12:18:21,390][INFO]: Deleting 88 rows 

## During the experimet

### Move files from server to the repo 

In [12]:

create_directory_structure(base_directory= cfg.DJ.userinfo.data_dir,
                           date=  cfg.data_subfolders.day, 
                           experiment = cfg.data_subfolders.experiment)

copy_stim_files(
    recording_files_dir=cfg.paths.recording_files_dir,  # type: ignore
    destination_base=cfg.DJ.userinfo.data_dir,  # type: ignore
    date=cfg.data_subfolders.day,  # type: ignore
    experiment=cfg.data_subfolders.experiment,  # type: ignore
    full_dummy_ini_dir= os.path.join(cfg.paths.repo_directory, cfg.paths.dummy_ini_dir),  # type: ignore
)

SKIPPING File /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/data/recordings/first_closed_loop_experiment/20250717/1/Raw/M1_RR_GCL1_MB_iter0.smh already exists, skipping.
SKIPPING File /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/data/recordings/first_closed_loop_experiment/20250717/1/Raw/M1_RR_GCL1_dn_iter0.smh already exists, skipping.
SKIPPING File /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/data/recordings/first_closed_loop_experiment/20250717/1/Raw/M1_RR_GCL1_dn_iter0.smp already exists, skipping.
SKIPPING File /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/data/recordings/first_closed_loop_experiment/20250717/1/Raw/M1_RR_GCL1_chirp_iter0.smh already exists, skipping.
SKIPPING File /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/data/recordings/first_closed_loop_experiment/20250717/1/Raw/M1_RR_GCL1_chirp_iter0.smp already exists, skipping.
SKIPPING File /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/data/recordi

In [16]:
preprocessor.upload_iteration_metadata()

Scanning for experimenter: closedlooptest
	header_path: /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/data/recordings/first_closed_loop_experiment/20250717/1
		header_name: 20250717_left.ini
		Adding: {'experimenter': 'closedlooptest', 'date': datetime.datetime(2025, 7, 17, 0, 0), 'exp_num': 1}


OpticDisk: 100%|██████████| 1/1 [00:00<00:00, 33.06it/s]


Found 3 files in 1 fields for key={'experimenter': 'closedlooptest', 'date': datetime.date(2025, 7, 17), 'exp_num': 1, 'raw_id': 1}
	Adding field: `{'field': 'GCL1', 'region': 'RR', 'cond1': 'iter0', 'experimenter': 'closedlooptest', 'date': datetime.date(2025, 7, 17), 'exp_num': 1, 'raw_id': 1}`


Processes: 100%|██████████| 6/6 [00:03<00:00,  1.55it/s]


## Analysis and Visualization

In [17]:
missing_keys = dj_table_holder("RoiMask")().list_missing_field()
field_key = missing_keys[0]
missing_keys
# import datetime
# test_key = {'experimenter': 'closedlooptest',
#   'date': datetime.date(2025, 7, 17),
#   'exp_num': 1,
#   'raw_id': 1,
#   'field': 'GCL1',
#   'region': 'RR',
#   'cond1': 'iter0'}
# missing_keys = [test_key]

[{'experimenter': 'closedlooptest',
  'date': datetime.date(2025, 7, 17),
  'exp_num': 1,
  'raw_id': 1,
  'field': 'GCL1',
  'region': 'RR',
  'cond1': 'iter0'}]

In [18]:
# somehow I get a circular import error if I import this at the top
from simulations.gui.integrated_autorois import InteractiveRoiCanvas

In [19]:
online_analysis_gui = InteractiveRoiCanvas(
    dj_table_holder=dj_table_holder,
    dj_preprocessor=preprocessor,
    all_dj_wrappers=[quality_type_analysis_wrapper,sta_wrapper],
    field_key=field_key,
    canvas_width=30,
    )

Load model weights for cpu from checkpoint /gpfs01/euler/data/Resources/AutoROIs/models/UNET_v0.1.0/dropout_and_aug_regul.ckpt using config /gpfs01/euler/data/Resources/AutoROIs/models/UNET_v0.1.0/sd_images.yaml


In [20]:
display(online_analysis_gui.start_gui()) 